In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def read_inspect_b_data(filename):
    """
    Read data from inspect_b_output.dat file.
    
    Format:
    - Line 1: time (single number)
    - Following lines: mode_index, radial_data1, radial_data2, ...
    - Pattern repeats for each time step
    
    Returns:
    - times: array of simulation times
    - data: list of 2D arrays, each containing [mode_indices, radial_data]
    """
    times = []
    all_data = []
    
    with open(filename, 'r') as f:
        lines = f.readlines()
    
    i = 0
    while i < len(lines):
        # Read time (single number line)
        time_line = lines[i].strip()
        try:
            time = float(time_line)
            times.append(time)
            i += 1
        except ValueError:
            print(f"Warning: Could not parse time from line {i}: {time_line}")
            i += 1
            continue
        
        # Read the first data line to determine the size
        if i >= len(lines):
            break
            
        first_data_line = lines[i].strip()
        first_data = [float(x.rstrip(',')) for x in first_data_line.split() if x.rstrip(',')]
        
        if len(first_data) < 2:
            print(f"Warning: Insufficient data in line {i}")
            continue
            
        # Determine number of radial points from first data line
        nrad = len(first_data) - 1  # First number is mode index
        
        # Read all mode data for this time step
        mode_data = []
        mode_indices = []
        
        while i < len(lines):
            data_line = lines[i].strip()
            
            # Try to parse as data line (mode_index + radial_data)
            try:
                data_values = [float(x.rstrip(',')) for x in data_line.split() if x.rstrip(',')]
                
                # Check if this is a single number (next time step)
                if len(data_values) == 1:
                    # This is the next time step, don't increment i
                    break
                
                # Check if this has the expected number of data points
                if len(data_values) == nrad + 1:  # mode_index + nrad data points
                    mode_index = int(data_values[0])
                    radial_data = np.array(data_values[1:])
                    
                    mode_indices.append(mode_index)
                    mode_data.append(radial_data)
                    i += 1
                else:
                    print(f"Warning: Unexpected data size in line {i}: expected {nrad+1}, got {len(data_values)}")
                    i += 1
                    
            except ValueError:
                print(f"Warning: Could not parse data from line {i}: {data_line}")
                i += 1
                break
        
        # Store the data for this time step
        if mode_data:
            mode_data_array = np.array(mode_data)  # Shape: (n_modes, n_radial)
            mode_indices_array = np.array(mode_indices)
            all_data.append({
                'mode_indices': mode_indices_array,
                'data': mode_data_array,
                'nrad': nrad
            })
    
    return np.array(times), all_data

def plot_data(times, all_data, mode_to_plot=1, radial_point=1, ax=None):
    """
    Plot time evolution of a specific radial point for a specific mode.
    
    Parameters:
    - times: array of simulation times
    - all_data: list of data dictionaries
    - mode_to_plot: azimuthal mode index to plot
    - radial_point: radial grid point index to plot (1-based)
    - ax: optional matplotlib axes to plot on (if None, creates new figure)
    
    Returns:
    - ax: matplotlib axes object (if data was plotted)
    """
    values = []
    valid_times = []
    
    for i, (time, data_dict) in enumerate(zip(times, all_data)):
        mode_indices = data_dict['mode_indices']
        data = data_dict['data']
        
        # Find the requested mode
        mode_mask = mode_indices == mode_to_plot
        if np.any(mode_mask):
            mode_data = data[mode_mask][0]  # Take first match
            if 0 <= radial_point-1 < len(mode_data):
                values.append(mode_data[radial_point-1])  # Convert to 0-based
                valid_times.append(time)
    
    if values:
        created_new_figure = False
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 6))
            created_new_figure = True
        
        # Add label to distinguish different lines
        ax.plot(valid_times, values, linewidth=2, label=f'Mode {mode_to_plot}, Radial {radial_point}')
        ax.set_xlabel('Simulation Time')
        ax.set_ylabel('Value')
        ax.set_title(f'Time Evolution Comparison')
        ax.grid(True, alpha=0.3)
        ax.legend()  # Add legend to show different lines
        
        if created_new_figure:
            plt.show()
        
        return ax
    else:
        print(f"No data found for mode {mode_to_plot}")
        return None

def plot_spectrum(times, all_data, mode_to_plot=1, time_indices=None):
    """
    Plot the spectrum of a specific mode as a function of radial points.
    
    Parameters:
    - times: array of simulation times
    - all_data: list of data dictionaries
    - mode_to_plot: azimuthal mode index to plot
    - time_indices: optional list of time indices to plot (if None, plots all times)
    """
    if time_indices is None:
        time_indices = range(len(times))
    
    plt.figure(figsize=(12, 8))
    
    for time_idx in time_indices:
        if time_idx >= len(all_data):
            print(f"Warning: time index {time_idx} out of range")
            continue
            
        data_dict = all_data[time_idx]
        mode_indices = data_dict['mode_indices']
        data = data_dict['data']
        
        # Find the requested mode
        mode_mask = mode_indices == mode_to_plot
        if np.any(mode_mask):
            mode_data = data[mode_mask][0]  # Take first match
            spectrum = np.abs(mode_data)
            
            # Create radial grid (assuming uniform spacing)
            radial_points = np.arange(len(spectrum))+1
            
            plt.plot(radial_points, spectrum, label=f't = {times[time_idx]:.3f}', 
                    linewidth=2, alpha=0.7)
    
    plt.xlabel('Radial Point Index')
    plt.ylabel('Power Spectrum')
    plt.title(f'Radial Spectrum of Mode {mode_to_plot}')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.yscale('log')  # Often useful for spectrum data
    plt.show()

def print_data_summary(times, all_data):
    """Print summary of the loaded data."""
    print(f"Total time steps: {len(times)}")
    if len(times) > 0:
        print(f"Time range: {times[0]:.6f} to {times[-1]:.6f}")
    
    if all_data:
        first_data = all_data[0]
        print(f"Number of radial points: {first_data['nrad']}")
        print(f"Available modes in first time step: {first_data['mode_indices']}")
        
        # Check consistency across time steps
        nrad_values = [data['nrad'] for data in all_data]
        if len(set(nrad_values)) > 1:
            print(f"Warning: Inconsistent number of radial points: {set(nrad_values)}")


In [ ]:
filename = "../../output/inspect_b_output.dat" 

try:
    times, all_data = read_inspect_b_data(filename)
    print("read complete")
    print_data_summary(times, all_data)
    
    # Plot example
    if all_data:
        # Plot mode 1, radial point 1
        plot_data(times, all_data, mode_to_plot=1, radial_point=1)
            
except FileNotFoundError:
    print(f"File {filename} not found!")
except Exception as e:
    print(f"Error reading file: {e}")

In [ ]:
time_indices_list = list(range(1, len(times), 50))  
plot_spectrum(times, all_data, mode_to_plot=1, time_indices=time_indices_list)  

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for i in range(5):
    plot_data(times, all_data, mode_to_plot=1, radial_point=i+1, ax=ax)
for i in range(340, 350):
    plot_data(times, all_data, mode_to_plot=1, radial_point=i+1, ax=ax)
plt.show()

In [ ]:
np.size(all_data[0]['data'])/10